# QT Fine-tuning on A100 (Colab Pro)

**Before running:** Change runtime to **A100 GPU**  
Runtime → Change runtime type → GPU → A100

This notebook:
1. Mounts Google Drive (upload zip files there first)
2. Installs dependencies
3. Extracts both PTB-XL datasets
4. Runs smoke test then full 30-epoch QT fine-tuning
5. Downloads the best checkpoint

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND')
print('CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Switch runtime to A100 first!'

## Step 1 – Upload datasets to Google Drive

Upload the following zip files to **MyDrive/ptbxl/** in Google Drive **before** running this cell:

```
ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1.zip
ptb-xl-a-comprehensive-electrocardiographic-feature-dataset-1.0.1.zip
AnchoredECG.zip   <-- zip of the AnchoredECG/ repo folder
shd_pretrained.pt  <-- upload to MyDrive/ptbxl/ as well (if you have it)
```

Create the zip from your Mac terminal:
```bash
cd "/Users/hakeemshindy/Downloads/ICML Supplementary"
zip -r ~/Desktop/AnchoredECG.zip AnchoredECG/ -x '*.pyc' -x '__pycache__/*' -x '*.DS_Store'
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess

DRIVE_DIR = '/content/drive/MyDrive/ptbxl'
WORK_DIR  = '/content/work'
os.makedirs(WORK_DIR, exist_ok=True)

def unzip_if_needed(zip_name, dest):
    src = f'{DRIVE_DIR}/{zip_name}'
    if not os.path.exists(dest):
        print(f'Extracting {zip_name} ...')
        subprocess.run(['unzip', '-q', src, '-d', WORK_DIR], check=True)
        print('  done.')
    else:
        print(f'  {dest} already exists, skipping.')

unzip_if_needed(
    'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1.zip',
    f'{WORK_DIR}/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1'
)
unzip_if_needed(
    'ptb-xl-a-comprehensive-electrocardiographic-feature-dataset-1.0.1.zip',
    f'{WORK_DIR}/ptb-xl-a-comprehensive-electrocardiographic-feature-dataset-1.0.1'
)
unzip_if_needed('AnchoredECG.zip', f'{WORK_DIR}/AnchoredECG')

# Copy pretrained checkpoint
ckpt_src = f'{DRIVE_DIR}/shd_pretrained.pt'
ckpt_dst = f'{WORK_DIR}/AnchoredECG/checkpoints/shd_pretrained.pt'
os.makedirs(os.path.dirname(ckpt_dst), exist_ok=True)
if os.path.exists(ckpt_src) and not os.path.exists(ckpt_dst):
    import shutil; shutil.copy(ckpt_src, ckpt_dst)
    print('Copied shd_pretrained.pt')
elif not os.path.exists(ckpt_src):
    print('WARNING: shd_pretrained.pt not found in Drive -- will train from scratch')

print('\nDirectory layout:')
for d in os.listdir(WORK_DIR):
    print(' ', d)

In [ ]:
# Install PyTorch Geometric (matches Colab's default torch)
import torch
TORCH = torch.__version__.split('+')[0]
CUDA  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'torch={TORCH}  cuda={CUDA}')

!pip install -q wfdb tqdm scipy
!pip install -q torch-geometric
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html

In [ ]:
SIGNALS_DIR  = f'{WORK_DIR}/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1'
FEATURES_DIR = f'{WORK_DIR}/ptb-xl-a-comprehensive-electrocardiographic-feature-dataset-1.0.1'
REPO_DIR     = f'{WORK_DIR}/AnchoredECG'
LABELS_CSV   = f'{REPO_DIR}/qt_finetune/qt_labels.csv'
PRETRAINED   = f'{REPO_DIR}/checkpoints/shd_pretrained.pt'
OUTPUT_DIR   = f'{REPO_DIR}/qt_finetune/checkpoints'

import os
assert os.path.isdir(SIGNALS_DIR),  f'Missing: {SIGNALS_DIR}'
assert os.path.isdir(FEATURES_DIR), f'Missing: {FEATURES_DIR}'
assert os.path.isfile(LABELS_CSV),  f'Missing: {LABELS_CSV}'
print('All paths OK')

In [ ]:
# ---- Smoke test (loads 4 samples, one forward pass) ----------------------
%cd {REPO_DIR}

!python qt_finetune/finetune_qt.py \
    --ptbxl-dir        {SIGNALS_DIR} \
    --ptbxl-features-dir {FEATURES_DIR} \
    --labels-csv       {LABELS_CSV} \
    --checkpoint       {PRETRAINED} \
    --smoke-test

In [ ]:
# ---- Full 30-epoch fine-tuning (regression HEAD only; encoder frozen) --------
# On an A100 with batch=32 this takes ~15-25 min (head-only is faster).
# Add --unfreeze-encoder --encoder-lr 5e-5 to also train the GNN trunk.

%cd {REPO_DIR}

!python qt_finetune/finetune_qt.py \
    --ptbxl-dir          {SIGNALS_DIR} \
    --ptbxl-features-dir {FEATURES_DIR} \
    --labels-csv         {LABELS_CSV} \
    --checkpoint         {PRETRAINED} \
    --output-dir         {OUTPUT_DIR} \
    --epochs    30 \
    --batch-size 32 \
    --head-lr    1e-3 \
    --num-workers 4

In [ ]:
# ---- Copy best checkpoint back to Drive ----------------------------------
import shutil, os

src = f'{OUTPUT_DIR}/best_qt_model.pt'
dst = f'{DRIVE_DIR}/best_qt_model.pt'

if os.path.exists(src):
    shutil.copy(src, dst)
    print(f'Saved to Drive: {dst}')
else:
    print('best_qt_model.pt not found -- did training finish?')

In [ ]:
# ---- OR: download directly to your browser -------------------------------
from google.colab import files
files.download(f'{OUTPUT_DIR}/best_qt_model.pt')